In [1]:
from dotenv import load_dotenv
import os
load_dotenv('.env')
from langchain_core.messages import SystemMessage,HumanMessage

### Prompting Methods


In [2]:
# from langchain_google_genai import ChatGoogleGenerativeAI
# api_key = os.getenv('GOOGLE_API_KEY')
# Gllm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", google_api_key=api_key)

In [4]:
from langchain_groq import ChatGroq
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=os.getenv("GROQ_API_KEY"),
    max_tokens=1000
)

In [5]:
mes=[
    SystemMessage(content="You are a python developer"),
    HumanMessage(content="Write a calculator code")
]
res=llm.invoke(mes)
print(res.content)

**Calculator Code in Python**

Here's a basic calculator code in Python that performs addition, subtraction, multiplication, and division.

```python
# Calculator Code in Python

def add(x, y):
    """
    Adds two numbers.
    
    Args:
        x (float): The first number.
        y (float): The second number.
    
    Returns:
        float: The sum of x and y.
    """
    return x + y

def subtract(x, y):
    """
    Subtracts two numbers.
    
    Args:
        x (float): The first number.
        y (float): The second number.
    
    Returns:
        float: The difference of x and y.
    """
    return x - y

def multiply(x, y):
    """
    Multiplies two numbers.
    
    Args:
        x (float): The first number.
        y (float): The second number.
    
    Returns:
        float: The product of x and y.
    """
    return x * y

def divide(x, y):
    """
    Divides two numbers.
    
    Args:
        x (float): The dividend.
        y (float): The divisor.
    
    Returns

### PromptTemplates - Dynamic Prompts

In [6]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

out=StrOutputParser()

# FIXED: Use tuples instead of SystemMessage/HumanMessage objects
prompts = ChatPromptTemplate.from_messages([
    ("system", "You are a translator and translate the given input in {language}"),
    ("human", "{query}")
])
prompts

ChatPromptTemplate(input_variables=['language', 'query'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['language'], input_types={}, partial_variables={}, template='You are a translator and translate the given input in {language}'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, template='{query}'), additional_kwargs={})])

In [7]:
# DEBUG: Let's see what the formatted messages actually contain
final_prompt = prompts.format_messages(
    language="Hindi",
    query="I love to play online mobile games and watch good movies and series but for future i need to study......"
)

print("DEBUG: Formatted messages:")
for i, msg in enumerate(final_prompt):
    print(f"{i+1}. {msg.type}: {repr(msg.content)}")
print()

res = llm.invoke(final_prompt)
print("RESPONSE:")
print(res.content)

DEBUG: Formatted messages:
1. system: 'You are a translator and translate the given input in Hindi'
2. human: 'I love to play online mobile games and watch good movies and series but for future i need to study......'

RESPONSE:
मैं ऑनलाइन मोबाइल गेम्स खेलने और अच्छी फिल्में और श्रृंखलाएं देखने का शौकीन हूँ, लेकिन भविष्य के लिए मुझे पढ़ाई करनी होगी।


In [ ]:
# ALTERNATIVE: Using SystemMessage & HumanMessage with dynamic content
from langchain_core.prompts import PromptTemplate

# Create templates for each part
system_template = PromptTemplate.from_template("You are a translator and translate the given input in {language}")
human_template = PromptTemplate.from_template("{query}")

# Format them separately
system_message = SystemMessage(content=system_template.format(language="Hindi"))
human_message = HumanMessage(content=human_template.format(query="I love to play online mobile games..."))

# Use them directly with LLM
messages = [system_message, human_message]
res = Gllm.invoke(messages)
print("Using SystemMessage & HumanMessage objects:")
print(res.content)

In [ ]:
# RECOMMENDED: Use tuples for templates, message objects for direct use
from langchain_core.prompts import ChatPromptTemplate

# For dynamic prompts: Use tuples (allows variable substitution)
dynamic_prompt = ChatPromptTemplate.from_messages([
    ("system", "Translate to {language}: {query}"),
    ("human", "Please provide the translation.")
])

# For static prompts: Use message objects directly (easier to read)
static_messages = [
    SystemMessage(content="You are a helpful coding assistant."),
    HumanMessage(content="Write a Python function to calculate factorial.")
]

print("✅ You CAN use SystemMessage & HumanMessage objects!")
print("✅ Just not directly in ChatPromptTemplate.from_messages()")
print("✅ Use them for static content or after formatting templates")

In [ ]:
res=Gllm.invoke(prompts.invoke({"language":"Hindi",
    "query":"I love to play online mobile games and watch good movies and series but for future i need to study......"}))
print(res.content)

In [ ]:
chains=prompts | Gllm
res=chains.invoke({"language":"Hinglish",
    "query":"I love to play online mobile games and watch good movies and series but for future i need to study......"})
print(res.content)

In [33]:
from langchain_openai import ChatOpenAI

Ollm=ChatOpenAI(
    model="openai/gpt-4o",
    api_key=os.getenv("OPENROUTER_GPT_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
    max_tokens=1000
)


In [34]:
chains= prompts | Ollm | out
res=chains.invoke({"language":"Hinglish",
    "query":"I love to play online mobile games and watch good movies and series but for future i need to study......"})
print(res)

Mujhe online mobile games khelna aur achchi movies aur series dekhna pasand hai, lekin future ke liye mujhe padhai karni padegi...
